# YOLO11 classification setup

Validasi dataset dan muat checkpoint klasifikasi YOLO11 tanpa memulai training.

In [ ]:
from pathlib import Path

from ultralytics import YOLO

EXPECTED_CLASSES = {"adidas", "converse", "nike"}
REQUIRED_SPLITS = ("train", "val", "test")
EXPECTED_COUNTS_PER_CLASS = {"train": 193, "val": 55, "test": 27}
MODEL_CHECKPOINT = "yolo11n-cls.pt"
SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DATASET_ROOT = Path("../datasets/shoe_brands_stratified").resolve()

Struktur minimum dataset:

```text
shoe_brands/
├── train/{adidas,converse,nike}/
└── val/{adidas,converse,nike}/
```

In [ ]:
def count_images(directory: Path) -> int:
    return sum(
        1
        for file_path in directory.iterdir()
        if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
    )


def validate_dataset(dataset_root: Path) -> dict[str, dict[str, int]]:
    missing_directories = [
        dataset_root / split_name / class_name
        for split_name in REQUIRED_SPLITS
        for class_name in sorted(EXPECTED_CLASSES)
        if not (dataset_root / split_name / class_name).is_dir()
    ]

    if missing_directories:
        formatted_paths = "\n".join(f"  - {path}" for path in missing_directories)
        raise FileNotFoundError(
            "Dataset structure is incomplete. Create these directories:\n"
            f"{formatted_paths}"
        )

    image_counts = {
        split_name: {
            class_name: count_images(dataset_root / split_name / class_name)
            for class_name in sorted(EXPECTED_CLASSES)
        }
        for split_name in REQUIRED_SPLITS
    }
    invalid_counts = [
        f"{split_name}/{class_name}: {image_count}"
        for split_name, class_counts in image_counts.items()
        for class_name, image_count in class_counts.items()
        if image_count != EXPECTED_COUNTS_PER_CLASS[split_name]
    ]
    if invalid_counts:
        raise ValueError(f"Unexpected image counts: {invalid_counts}")

    return image_counts


image_counts = validate_dataset(DATASET_ROOT)
print(f"Dataset ready: {DATASET_ROOT}")
print(f"Classes: {', '.join(sorted(EXPECTED_CLASSES))}")
for split_name, class_counts in image_counts.items():
    print(f"{split_name}: {class_counts}")

In [ ]:
model = YOLO(MODEL_CHECKPOINT)
print(f"YOLO11 checkpoint ready: {model.ckpt_path}")
print("Setup complete. Training was not started.")

## Referensi konfigurasi training (tidak dijalankan)

Saat dataset sudah siap dan training disetujui, konfigurasi awal dapat menggunakan `model.train(data=DATASET_ROOT, epochs=..., imgsz=...)`. Notebook ini sengaja tidak memiliki cell yang memanggil training.